# Train sLSTM on Kaggle – MOT17 Trajectory Prediction

Notebook này train **sLSTM only** (`xLSTM[0:1]`) trên tập **MOT17** theo bài toán dự đoán trajectory.

Mỗi bounding box (x_c, y_c, w, h) được lượng tử hoá thành 4 token rời rạc.  
Model học dự đoán token tiếp theo trong chuỗi toạ độ của từng track.

**GPU Kaggle T4 (CC=7.5):** tự động dùng `backend: vanilla` vì sLSTM CUDA kernel yêu cầu CC >= 8.0.

## 1. Clone repo & cài dependencies

In [10]:
import os

REPO_DIR = "/kaggle/working/xlstm"

if not os.path.exists(REPO_DIR):
    os.system("git clone https://github.com/NX-AI/xlstm.git " + REPO_DIR)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

Working dir: /kaggle/working/xlstm


In [11]:
!pip install -q dacite omegaconf torchmetrics einops
!pip install -q -e . --no-deps
print("Done installing.")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for xlstm (pyproject.toml) ... done
Done installing.


## 2. Kiểm tra GPU

In [12]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    cc = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {cc[0]}.{cc[1]}")
    if cc[0] < 8:
        print("=> CC < 8.0: se dung backend=vanilla")
    else:
        print("=> CC >= 8.0: se dung backend=cuda")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
Compute Capability: 7.5
=> CC < 8.0: se dung backend=vanilla


## 3. Tạo config sLSTM

In [13]:
import os
import torch

REPO_DIR = "/kaggle/working/xlstm"
if os.getcwd() != REPO_DIR:
    os.chdir(REPO_DIR)

if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    slstm_backend = "cuda"
else:
    slstm_backend = "vanilla"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Backend:", slstm_backend)
print("Device :", device)

# Giữ nguyên tất cả thông số training & model so với bản gốc.
# Chỉ cập nhật vocab_size=256 (8-bit quantization cho toạ độ) và
# context_length=256 (64 frames × 4 token/frame) cho phù hợp dataset MOT17.
config_yaml = f"""training:
  batch_size: 256
  lr: 0.001
  seed: 42
  val_every_step: 200
  lr_warmup_steps: 2000
  lr_decay_until_steps: ${{.num_steps}}
  lr_decay_factor: 0.001
  weight_decay: 0.1
  num_steps: 20000
  device: {device}
  amp_precision: bfloat16
  weight_precision: float32
  enable_mixed_precision: true

model:
  num_blocks: 2
  embedding_dim: 64
  mlstm_block:
    mlstm:
      num_heads: 1
  slstm_block:
    slstm:
      backend: {slstm_backend}
      num_heads: 1
  slstm_at: [0, 1]
  context_length: 256
  vocab_size: 256

dataset:
  data_root: /kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train
  vocab_size: 256
  context_length: 256
  min_track_length: 10
  val_ratio: 0.1
"""

os.makedirs("experiments", exist_ok=True)
with open("experiments/slstm_kaggle.yaml", "w") as f:
    f.write(config_yaml)

print("Config saved -> experiments/slstm_kaggle.yaml")
print(config_yaml)

Backend: vanilla
Device : cuda
Config saved -> experiments/slstm_kaggle.yaml
training:
  batch_size: 256
  lr: 0.001
  seed: 42
  val_every_step: 200
  lr_warmup_steps: 2000
  lr_decay_until_steps: ${.num_steps}
  lr_decay_factor: 0.001
  weight_decay: 0.1
  num_steps: 20000
  device: cuda
  amp_precision: bfloat16
  weight_precision: float32
  enable_mixed_precision: true

model:
  num_blocks: 2
  embedding_dim: 64
  mlstm_block:
    mlstm:
      num_heads: 1
  slstm_block:
    slstm:
      backend: vanilla
      num_heads: 1
  slstm_at: [0, 1]
  context_length: 256
  vocab_size: 256

dataset:
  data_root: /kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train
  vocab_size: 256
  context_length: 256
  min_track_length: 10
  val_ratio: 0.1



## 4. MOT17 Dataset – Trajectory Tokenizer

In [14]:
import configparser
import numpy as np
import torch
from torch.utils.data import Dataset


# MOT17-11 và MOT17-13 giữ lại làm test set (không tham gia training)
TRAIN_SEQS = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
]
TEST_SEQS = [
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]


class MOT17TrajectoryDataset(Dataset):
    """Mỗi sample là chuỗi token (context_length,) và target (context_length,).

    Mỗi bounding box → 4 token [x_c, y_c, w, h] lượng tử hoá vào [0, vocab_size-1].
    Model được train dự đoán token tiếp theo (causal LM style).
    """

    def __init__(self, data_root, sequences, context_length, vocab_size,
                 min_track_length=10, val_ratio=0.1, split="train", seed=42):
        self.context_length = context_length
        self.vocab_size = vocab_size

        all_flat_tracks = []

        for seq in sequences:
            seq_path  = os.path.join(data_root, seq)
            gt_path   = os.path.join(seq_path, "gt", "gt.txt")
            info_path = os.path.join(seq_path, "seqinfo.ini")

            if not os.path.exists(gt_path):
                print(f"[WARN] Bỏ qua {seq}: không tìm thấy gt.txt")
                continue

            cfg_ini = configparser.ConfigParser()
            cfg_ini.read(info_path)
            img_w = int(cfg_ini["Sequence"]["imWidth"])
            img_h = int(cfg_ini["Sequence"]["imHeight"])

            gt = np.loadtxt(gt_path, delimiter=",")
            # columns: frame, id, left, top, w, h, conf, class, visibility
            # Giữ lại: pedestrian (class=1), được annotate (conf=1)
            mask = (gt[:, 7] == 1) & (gt[:, 6] == 1)
            gt = gt[mask]

            for tid in np.unique(gt[:, 1]):
                trk = gt[gt[:, 1] == tid]
                trk = trk[np.argsort(trk[:, 0])]

                if len(trk) < min_track_length:
                    continue

                # Chuyển sang x_c, y_c, w, h chuẩn hoá về [0,1]
                left, top, w, h = trk[:, 2], trk[:, 3], trk[:, 4], trk[:, 5]
                x_c = np.clip((left + w / 2) / img_w, 0.0, 1.0)
                y_c = np.clip((top  + h / 2) / img_h, 0.0, 1.0)
                w_n = np.clip(w / img_w, 0.0, 1.0)
                h_n = np.clip(h / img_h, 0.0, 1.0)

                # Lượng tử hoá → token [0, vocab_size-1]
                V = vocab_size - 1
                tokens = np.stack([
                    np.floor(x_c * V).astype(np.int64),
                    np.floor(y_c * V).astype(np.int64),
                    np.floor(w_n * V).astype(np.int64),
                    np.floor(h_n * V).astype(np.int64),
                ], axis=1)  # (T, 4)
                flat = np.clip(tokens, 0, V).flatten()  # (T*4,)
                all_flat_tracks.append(flat)

        # Tạo segments bằng sliding window
        stride = max(1, context_length // 4)
        segments = []
        for flat in all_flat_tracks:
            for start in range(0, len(flat) - context_length, stride):
                seg = flat[start : start + context_length + 1]
                if len(seg) == context_length + 1:
                    segments.append(seg.copy())

        rng = np.random.default_rng(seed)
        segments = np.array(segments, dtype=np.int64)
        idx = rng.permutation(len(segments))
        segments = segments[idx]

        n_val = max(1, int(len(segments) * val_ratio))
        if split == "train":
            self.segments = segments[n_val:]
        else:
            self.segments = segments[:n_val]

        print(f"[{split}] {len(self.segments)} segments từ {len(all_flat_tracks)} tracks")

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        seg = torch.from_numpy(self.segments[idx])
        return seg[:-1], seg[1:]


print("MOT17TrajectoryDataset class defined.")

MOT17TrajectoryDataset class defined.


## 5. Load dataset & khởi tạo model

In [15]:
import sys
sys.path.insert(0, ".")

import torch
import torch.optim as optim
from dacite import from_dict
from omegaconf import OmegaConf
from torch import nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from xlstm.xlstm_lm_model import xLSTMLMModel, xLSTMLMModelConfig
from experiments.lr_scheduler import LinearWarmupCosineAnnealing

torch_dtype_map = {
    "float32" : torch.float32,
    "bfloat16": torch.bfloat16,
    "float16" : torch.float16,
}

with open("experiments/slstm_kaggle.yaml", "r") as f:
    cfg = OmegaConf.create(f.read())
OmegaConf.resolve(cfg)

print(OmegaConf.to_yaml(cfg))

training:
  batch_size: 256
  lr: 0.001
  seed: 42
  val_every_step: 200
  lr_warmup_steps: 2000
  lr_decay_until_steps: 20000
  lr_decay_factor: 0.001
  weight_decay: 0.1
  num_steps: 20000
  device: cuda
  amp_precision: bfloat16
  weight_precision: float32
  enable_mixed_precision: true
model:
  num_blocks: 2
  embedding_dim: 64
  mlstm_block:
    mlstm:
      num_heads: 1
  slstm_block:
    slstm:
      backend: vanilla
      num_heads: 1
  slstm_at:
  - 0
  - 1
  context_length: 256
  vocab_size: 256
dataset:
  data_root: /kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train
  vocab_size: 256
  context_length: 256
  min_track_length: 10
  val_ratio: 0.1



In [16]:
torch.manual_seed(cfg.training.seed)

print("Đang load MOT17 dataset...")
train_dataset = MOT17TrajectoryDataset(
    data_root        = cfg.dataset.data_root,
    sequences        = TRAIN_SEQS,
    context_length   = cfg.dataset.context_length,
    vocab_size       = cfg.dataset.vocab_size,
    min_track_length = cfg.dataset.min_track_length,
    val_ratio        = cfg.dataset.val_ratio,
    split            = "train",
)

val_dataset = MOT17TrajectoryDataset(
    data_root        = cfg.dataset.data_root,
    sequences        = TRAIN_SEQS,
    context_length   = cfg.dataset.context_length,
    vocab_size       = cfg.dataset.vocab_size,
    min_track_length = cfg.dataset.min_track_length,
    val_ratio        = cfg.dataset.val_ratio,
    split            = "val",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.training.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Đang load MOT17 dataset...
[train] 4212 segments từ 333 tracks
[val] 467 segments từ 333 tracks
Train batches: 16 | Val batches: 2


In [17]:
print("Khởi tạo model sLSTM...")
model = xLSTMLMModel(
    from_dict(xLSTMLMModelConfig, OmegaConf.to_container(cfg.model))
).to(device=cfg.training.device)
model.reset_parameters()
model = model.to(dtype=torch_dtype_map[cfg.training.weight_precision])

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

Khởi tạo model sLSTM...
Model parameters: 149,056
xLSTMLMModel(
  (xlstm_block_stack): xLSTMBlockStack(
    (blocks): ModuleList(
      (0-1): 2 x sLSTMBlock(
        (xlstm_norm): LayerNorm()
        (xlstm): sLSTMLayer(
          (conv1d): CausalConv1d(
            (conv): Conv1d(64, 64, kernel_size=(4,), stride=(1,), padding=(3,), groups=64)
          )
          (conv_act_fn): SiLU()
          (fgate): LinearHeadwiseExpand(in_features=64, num_heads=1, expand_factor_up=1, bias=False, trainable_weight=True, trainable_bias=True, )
          (igate): LinearHeadwiseExpand(in_features=64, num_heads=1, expand_factor_up=1, bias=False, trainable_weight=True, trainable_bias=True, )
          (zgate): LinearHeadwiseExpand(in_features=64, num_heads=1, expand_factor_up=1, bias=False, trainable_weight=True, trainable_bias=True, )
          (ogate): LinearHeadwiseExpand(in_features=64, num_heads=1, expand_factor_up=1, bias=False, trainable_weight=True, trainable_bias=True, )
          (slstm_cell

## 6. Train sLSTM

In [ ]:
optim_groups = model._create_weight_decay_optim_groups()
optimizer = optim.AdamW(
    [
        {"weight_decay": cfg.training.weight_decay, "params": optim_groups[0]},
        {"weight_decay": 0.0,                       "params": optim_groups[1]},
    ],
    lr=cfg.training.lr,
)
lr_scheduler = LinearWarmupCosineAnnealing(
    optimizer,
    cfg.training.lr_warmup_steps,
    cfg.training.lr_decay_until_steps,
    cfg.training.lr,
    cfg.training.lr_decay_factor * cfg.training.lr,
)

step         = 0
epoch        = 1
running_loss = 0.0
history      = {"step": [], "train_loss": [], "val_loss": []}

print(f"Bắt đầu training: {cfg.training.num_steps} steps | device={cfg.training.device}")

while step < cfg.training.num_steps:
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for inputs, labels in pbar:
        inputs = inputs.to(cfg.training.device)
        labels = labels.to(cfg.training.device)

        model.train()
        optimizer.zero_grad()
        with torch.autocast(
            device_type=cfg.training.device,
            dtype=torch_dtype_map[cfg.training.amp_precision],
            enabled=cfg.training.enable_mixed_precision,
        ):
            outputs = model(inputs)
            loss = nn.functional.cross_entropy(
                outputs.view(-1, cfg.model.vocab_size),
                labels.view(-1),
            )

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        running_loss = running_loss * step / (step + 1) + loss.item() / (step + 1)
        step += 1
        pbar.set_postfix({"loss": f"{running_loss:.4f}", "step": step})

        if step % cfg.training.val_every_step == 0:
            history["step"].append(step)
            history["train_loss"].append(running_loss)

            model.eval()
            val_loss_total = 0.0
            val_correct    = 0
            val_total      = 0
            with torch.no_grad():
                for vi, vl in val_loader:
                    vi = vi.to(cfg.training.device)
                    vl = vl.to(cfg.training.device)
                    with torch.autocast(
                        device_type=cfg.training.device,
                        dtype=torch_dtype_map[cfg.training.amp_precision],
                        enabled=cfg.training.enable_mixed_precision,
                    ):
                        vo = model(vi)
                        val_loss_total += nn.functional.cross_entropy(
                            vo.view(-1, cfg.model.vocab_size),
                            vl.view(-1),
                        ).item()
                    preds = vo.argmax(dim=-1)
                    val_correct += (preds == vl).sum().item()
                    val_total   += vl.numel()

            avg_val = val_loss_total / len(val_loader)
            val_acc = val_correct / val_total
            history["val_loss"].append(avg_val)

            print(f"\n[Step {step}/{cfg.training.num_steps}] "
                  f"Train Loss: {running_loss:.4f} | "
                  f"Val Loss: {avg_val:.4f} | "
                  f"Val Token Acc: {val_acc:.4f}")

        if step >= cfg.training.num_steps:
            break
    epoch += 1

print("\nTraining xong!")

Bắt đầu training: 20000 steps | device=cuda


Epoch 1:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/16 [00:00<?, ?it/s]

...

Epoch 74:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 75:   0%|          | 0/16 [00:00<?, ?it/s]


[Step 1200/20000] Train Loss: 2.5560 | Val Loss: 0.6046 | Val Token Acc: 0.8453


Epoch 76:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch 77:   0%|          | 0/16 [00:00<?, ?it/s]

## 7. Vẽ loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history["step"], history["train_loss"], label="Train Loss", marker="o")
if history["val_loss"]:
    plt.plot(history["step"][:len(history["val_loss"])], history["val_loss"],
             label="Val Loss", marker="s")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("sLSTM Training – MOT17 Trajectory Prediction")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Đánh giá ADE / FDE trên test sequences

Dùng **MOT17-11** và **MOT17-13** (không tham gia training) để đánh giá.

- **Context**: 52 frame đầu (= 208 token) → làm input cho model  
- **Prediction**: model tự sinh 12 frame tiếp theo (= 48 token) theo kiểu auto-regressive (greedy)
- **ADE** (Average Displacement Error): trung bình L2 distance giữa tâm bbox dự đoán và thực tế, qua tất cả 12 frame dự đoán (đơn vị pixel)
- **FDE** (Final Displacement Error): L2 distance ở frame thứ 12 (đơn vị pixel)

In [ ]:
# ── Thông số đánh giá ──────────────────────────────────────────────────────────
CONTEXT_TOKENS = 208   # 52 frame × 4 token — input cho model
PRED_TOKENS    = 48    # 12 frame × 4 token — model phải tự sinh
MIN_TRACK_EVAL = (CONTEXT_TOKENS + PRED_TOKENS) // 4  # track ngắn hơn bỏ qua


def decode_coord(token_arr, vocab_size=256):
    """Token [0, vocab_size-1] → toạ độ chuẩn hoá [0, 1]"""
    return token_arr / (vocab_size - 1)


def tokenize_track(trk, img_w, img_h, vocab_size=256):
    """ndarray (T,9) gt rows → flat token array (T*4,)"""
    left, top, w, h = trk[:, 2], trk[:, 3], trk[:, 4], trk[:, 5]
    V   = vocab_size - 1
    x_c = np.clip((left + w / 2) / img_w, 0.0, 1.0)
    y_c = np.clip((top  + h / 2) / img_h, 0.0, 1.0)
    w_n = np.clip(w / img_w, 0.0, 1.0)
    h_n = np.clip(h / img_h, 0.0, 1.0)
    tokens = np.stack([
        np.floor(x_c * V).astype(np.int64),
        np.floor(y_c * V).astype(np.int64),
        np.floor(w_n * V).astype(np.int64),
        np.floor(h_n * V).astype(np.int64),
    ], axis=1)
    return np.clip(tokens, 0, V).flatten()


model.eval()
ade_all, fde_all = [], []

for seq in TEST_SEQS:
    seq_path  = os.path.join(cfg.dataset.data_root, seq)
    gt_path   = os.path.join(seq_path, "gt", "gt.txt")
    info_path = os.path.join(seq_path, "seqinfo.ini")

    cfg_ini = configparser.ConfigParser()
    cfg_ini.read(info_path)
    img_w = int(cfg_ini["Sequence"]["imWidth"])
    img_h = int(cfg_ini["Sequence"]["imHeight"])

    gt = np.loadtxt(gt_path, delimiter=",")
    gt = gt[(gt[:, 7] == 1) & (gt[:, 6] == 1)]

    seq_ade, seq_fde = [], []

    for tid in np.unique(gt[:, 1]):
        trk = gt[gt[:, 1] == tid]
        trk = trk[np.argsort(trk[:, 0])]

        if len(trk) < MIN_TRACK_EVAL:
            continue

        flat = tokenize_track(trk, img_w, img_h, cfg.dataset.vocab_size)

        # Input: CONTEXT_TOKENS token đầu tiên của track
        ctx = torch.tensor(flat[:CONTEXT_TOKENS], dtype=torch.long)
        ctx = ctx.unsqueeze(0).to(cfg.training.device)  # (1, 208)

        # Auto-regressive sinh từng token một (greedy decoding)
        generated = []
        with torch.no_grad():
            for _ in range(PRED_TOKENS):
                with torch.autocast(
                    device_type=cfg.training.device,
                    dtype=torch_dtype_map[cfg.training.amp_precision],
                    enabled=cfg.training.enable_mixed_precision,
                ):
                    out = model(ctx)                    # (1, T, vocab_size)
                next_tok = out[0, -1].argmax(-1)        # greedy: lấy token có prob cao nhất
                generated.append(next_tok.item())
                ctx = torch.cat([ctx, next_tok.view(1, 1)], dim=1)

        # Giải mã token về toạ độ pixel
        pred_t = np.array(generated, dtype=np.float32).reshape(-1, 4)   # (12, 4)
        true_t = flat[CONTEXT_TOKENS:CONTEXT_TOKENS + PRED_TOKENS].astype(np.float32).reshape(-1, 4)

        pred_xc = decode_coord(pred_t[:, 0], cfg.dataset.vocab_size) * img_w
        pred_yc = decode_coord(pred_t[:, 1], cfg.dataset.vocab_size) * img_h
        true_xc = decode_coord(true_t[:, 0], cfg.dataset.vocab_size) * img_w
        true_yc = decode_coord(true_t[:, 1], cfg.dataset.vocab_size) * img_h

        dist = np.sqrt((pred_xc - true_xc) ** 2 + (pred_yc - true_yc) ** 2)  # (12,)
        seq_ade.append(float(dist.mean()))
        seq_fde.append(float(dist[-1]))

    if seq_ade:
        print(f"{seq:20s}  ADE={np.mean(seq_ade):7.2f} px  "
              f"FDE={np.mean(seq_fde):7.2f} px  "
              f"(tracks đánh giá: {len(seq_ade)})")
    ade_all.extend(seq_ade)
    fde_all.extend(seq_fde)

print(f"\n{'─'*55}")
print(f"Tổng hợp ({len(ade_all)} tracks từ {len(TEST_SEQS)} sequences)")
print(f"  ADE : {np.mean(ade_all):.2f} px")
print(f"  FDE : {np.mean(fde_all):.2f} px")

### Visualise: so sánh trajectory dự đoán vs thực tế (5 track đầu tiên)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Lấy seq đầu tiên trong TEST_SEQS để visualise
VIS_SEQ   = TEST_SEQS[0]
VIS_LIMIT = 5   # số track vẽ

seq_path  = os.path.join(cfg.dataset.data_root, VIS_SEQ)
gt_path   = os.path.join(seq_path, "gt", "gt.txt")
info_path = os.path.join(seq_path, "seqinfo.ini")

cfg_ini = configparser.ConfigParser()
cfg_ini.read(info_path)
img_w = int(cfg_ini["Sequence"]["imWidth"])
img_h = int(cfg_ini["Sequence"]["imHeight"])

gt = np.loadtxt(gt_path, delimiter=",")
gt = gt[(gt[:, 7] == 1) & (gt[:, 6] == 1)]

fig, axes = plt.subplots(1, VIS_LIMIT, figsize=(VIS_LIMIT * 4, 4))
plotted = 0

model.eval()
for tid in np.unique(gt[:, 1]):
    if plotted >= VIS_LIMIT:
        break
    trk = gt[gt[:, 1] == tid]
    trk = trk[np.argsort(trk[:, 0])]
    if len(trk) < MIN_TRACK_EVAL:
        continue

    flat = tokenize_track(trk, img_w, img_h, cfg.dataset.vocab_size)
    ctx  = torch.tensor(flat[:CONTEXT_TOKENS], dtype=torch.long).unsqueeze(0).to(cfg.training.device)

    generated = []
    with torch.no_grad():
        for _ in range(PRED_TOKENS):
            with torch.autocast(
                device_type=cfg.training.device,
                dtype=torch_dtype_map[cfg.training.amp_precision],
                enabled=cfg.training.enable_mixed_precision,
            ):
                out = model(ctx)
            next_tok = out[0, -1].argmax(-1)
            generated.append(next_tok.item())
            ctx = torch.cat([ctx, next_tok.view(1, 1)], dim=1)

    pred_t = np.array(generated, dtype=np.float32).reshape(-1, 4)
    true_t = flat[CONTEXT_TOKENS:CONTEXT_TOKENS + PRED_TOKENS].astype(np.float32).reshape(-1, 4)
    ctx_t  = flat[:CONTEXT_TOKENS].astype(np.float32).reshape(-1, 4)

    # Giải mã tâm bbox về [0,1]
    V = cfg.dataset.vocab_size - 1
    ctx_xy  = ctx_t[:, :2]  / V
    pred_xy = pred_t[:, :2] / V
    true_xy = true_t[:, :2] / V

    ax = axes[plotted]
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)          # ảnh: trục y hướng xuống
    ax.plot(ctx_xy[:, 0],  ctx_xy[:, 1],  "b.-", label="Context",  linewidth=1, markersize=3)
    ax.plot(true_xy[:, 0], true_xy[:, 1], "g.-", label="GT",       linewidth=1.5, markersize=4)
    ax.plot(pred_xy[:, 0], pred_xy[:, 1], "r.-", label="Predicted", linewidth=1.5, markersize=4)
    ax.set_title(f"Track {int(tid)}", fontsize=9)
    ax.legend(fontsize=7)
    ax.set_aspect("equal", adjustable="box")
    plotted += 1

plt.suptitle(f"Trajectory Prediction – {VIS_SEQ}", fontsize=11)
plt.tight_layout()
plt.show()

## 9. Lưu model

In [ ]:
import os

save_path = "/kaggle/working/slstm_mot17.pt"
torch.save({
    "model_state_dict"    : model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "step"                : step,
    "train_loss"          : running_loss,
    "config"              : OmegaConf.to_container(cfg),
    "eval": {
        "ade_px": float(np.mean(ade_all)) if ade_all else None,
        "fde_px": float(np.mean(fde_all)) if fde_all else None,
        "test_seqs": TEST_SEQS,
    },
}, save_path)

print(f"Model đã lưu tại: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024:.1f} KB")
if ade_all:
    print(f"ADE: {np.mean(ade_all):.2f} px  |  FDE: {np.mean(fde_all):.2f} px")